# 04 · Comparación de modelos ML — evaluación cuantitativa del cualificador

**TFM Data Science e IA — Juan Alonso · Tutor: Julio Valero · Curso 2025-2026**

Este notebook compara tres aproximaciones para el sistema de cualificación de leads:

1. **Baseline manual** — filtro binario por reglas duras (lo que haría un comercial sin ML)
2. **Similitud coseno** — modelo en producción (features estructuradas + embedding semántico)
3. **XGBoost con Positive-Unlabeled Learning** — ranker supervisado como benchmark

El modelo evaluado es exactamente el mismo que usa la app Streamlit (`src/similitud.py`), que combina:
- 167 features estructuradas (numéricas, booleanas, categóricas one-hot)
- 384 dimensiones del embedding de `resumen_llm` (sentence-transformers/all-MiniLM-L6-v2)

**Total: 551 features por restaurante.** Score coseno en rango [-1, 1].

## 1 · Carga de la capa gold y librerías

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.similitud import construir_vectores

gold = pd.read_parquet(ROOT / "data/gold/gold_restaurantes_madrid.parquet")
print(f"Gold cargada: {len(gold)} restaurantes x {gold.shape[1]} columnas")
gold.head(2)

## 2 · Construcción del espacio de features

Usamos exactamente la función `construir_vectores` de `src/similitud.py`, la misma que se ejecuta en la app Streamlit. Incluye el embedding de `resumen_llm` (384 dim) concatenado con las 167 features estructuradas.

In [ ]:
modelo = construir_vectores(gold, use_embedding=True)
X = modelo.X
ids = gold["restaurante_id"]

print(f"Matriz de vectores: {X.shape[0]} restaurantes x {X.shape[1]} features\n")
print("Bloques de features:")
for bloque, cols in modelo.columnas_usadas.items():
    print(f"  {bloque}: {len(cols) if isinstance(cols, list) else cols}")

## 3 · Definición del ICP premium (5 seeds)

Usamos los mismos 5 seeds configurados como Demo 1 en la app Streamlit. Se amplió de 3 a 5 seeds tras el feedback del tutor: con muestras pequeñas, el test LOO es exigente y añadir seeds mejora significativamente la estimación del centroide.

In [ ]:
# IDs exactos de los 5 seeds premium (mismos que usa la app Streamlit)
# Usamos IDs en vez de nombres para evitar colisiones con restaurantes homónimos en el dataset.
SEEDS_PREMIUM_IDS = [
    "b4c626a9f914",  # Sagardi (Cortes)
    "5ee59f48bde1",  # Sandó (Palacio)
    "1d4eb6981e34",  # Dray Martina (Justicia)
    "99601101fac4",  # AskuaBarra (Cortes)
    "8f0ebbb4291d",  # Gioia (Justicia)
]

seeds_df = gold[gold["restaurante_id"].isin(SEEDS_PREMIUM_IDS)]
seeds_premium_ids = seeds_df["restaurante_id"].tolist()

print(f"Seeds premium ({len(seeds_premium_ids)}):")
print(seeds_df[["nombre", "barrio", "posicionamiento_precio", "ambiente"]].to_string(index=False))


## 4 · Modelo 1 · Similitud coseno

El vector-ICP es el centroide (media) de los seeds. El score de cada restaurante es el coseno entre su vector y el centroide, en rango [-1, 1] (1 = idéntico, -1 = opuesto). En la práctica los scores caen en [0.4, 0.85].

In [ ]:
def scores_coseno(X: np.ndarray, ids: pd.Series, seeds_ids: list) -> pd.Series:
    mask = ids.isin(seeds_ids).values
    if mask.sum() == 0:
        raise ValueError("Ninguna seed encontrada en ids")
    centroide = X[mask].mean(axis=0, keepdims=True)
    sims = cosine_similarity(centroide, X)[0]
    return pd.Series(sims, index=ids.values, name="score_coseno")

scores = scores_coseno(X, ids, seeds_premium_ids)
top10 = scores.drop(seeds_premium_ids).sort_values(ascending=False).head(10)
print("TOP 10 leads recomendados por coseno:")
for i, (rid, s) in enumerate(top10.items(), 1):
    nombre = gold.loc[gold["restaurante_id"] == rid, "nombre"].iloc[0]
    print(f"{i:2}. {s:.3f}  {nombre}")

## 5 · Modelo 2 · XGBoost con Positive-Unlabeled Learning

Con solo 5 positivos reales, planteamos el problema como *positive-unlabeled learning*: los seeds son positivos, muestreamos negativos aleatorios del resto del universo. Entrenamos un clasificador y ordenamos por probabilidad.

In [ ]:
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    from xgboost import XGBClassifier


def scores_xgboost(X: np.ndarray, ids: pd.Series, seeds_ids: list,
                   n_neg: int = 200, seed: int = 42) -> pd.Series:
    rng = np.random.RandomState(seed)
    all_idx = np.arange(len(ids))
    seed_idx = np.where(ids.isin(seeds_ids).values)[0]
    no_seed_idx = np.setdiff1d(all_idx, seed_idx)
    neg_idx = rng.choice(no_seed_idx, size=min(n_neg, len(no_seed_idx)), replace=False)

    X_train = np.vstack([X[seed_idx], X[neg_idx]])
    y_train = np.array([1] * len(seed_idx) + [0] * len(neg_idx))

    clf = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                        eval_metric="logloss", verbosity=0)
    clf.fit(X_train, y_train)
    probs = clf.predict_proba(X)[:, 1]
    return pd.Series(probs, index=ids.values, name="score_xgb")


probs = scores_xgboost(X, ids, seeds_premium_ids)
top10_xgb = probs.drop(seeds_premium_ids).sort_values(ascending=False).head(10)
print("TOP 10 leads recomendados por XGBoost:")
for i, (rid, p) in enumerate(top10_xgb.items(), 1):
    nombre = gold.loc[gold["restaurante_id"] == rid, "nombre"].iloc[0]
    print(f"{i:2}. p={p:.3f}  {nombre}")

## 6 · Baseline manual — filtro por reglas duras

Simulamos lo que haría un comercial sin ML: filtro binario por características del ICP (posicionamiento premium/alta_gama + web + presencia digital activa). Es determinista y no genera ranking.

In [ ]:
def scores_baseline(df: pd.DataFrame) -> pd.Series:
    mask = (
        df["posicionamiento_precio"].isin(["premium", "alta_gama"]) &
        (df["tiene_web"] == True) &
        (df["presencia_digital"] == "activa")
    )
    return pd.Series(mask.astype(int).values, index=df["restaurante_id"].values, name="score_baseline")


baseline_scores = scores_baseline(gold)
print(f"Restaurantes que pasan el filtro manual: {baseline_scores.sum()}")
print(f"De los cuales seeds: {baseline_scores[seeds_premium_ids].sum()}/{len(seeds_premium_ids)}")

## 7 · Evaluación Leave-One-Out (LOO)

Para cada seed: la apartamos, calculamos scores con las otras 4, y comprobamos si el seed apartado aparece en el top K del ranking (Recall@K).

**Interpretación:** con 5 seeds y ~1683 restaurantes en el universo, Recall@20 pide que el seed apartado quede en el top 1.2% (muy exigente). Recall@100 en el top 6% (más realista).

In [ ]:
def loo_recall(scoring_fn, X, ids, seeds_ids, k, es_baseline=False, **kwargs):
    hits = 0
    positions = []
    for sid in seeds_ids:
        otras = [s for s in seeds_ids if s != sid]
        if es_baseline:
            scores = scoring_fn(gold)
        else:
            scores = scoring_fn(X, ids, otras, **kwargs)
        scores = scores.drop(otras, errors="ignore")
        ranked = scores.sort_values(ascending=False, kind="mergesort")
        try:
            pos = list(ranked.index).index(sid) + 1
            positions.append(pos)
            if pos <= k:
                hits += 1
        except ValueError:
            positions.append(len(ranked) + 1)
    return hits / len(seeds_ids), positions


print("=== EVALUACION LEAVE-ONE-OUT (5 seeds premium) ===\n")

for K_eval in [20, 100]:
    print(f"--- Recall@{K_eval} ---")
    r_baseline, pos_b = loo_recall(scores_baseline, X, ids, seeds_premium_ids, K_eval, es_baseline=True)
    r_coseno, pos_c = loo_recall(scores_coseno, X, ids, seeds_premium_ids, K_eval)
    r_xgb, pos_x = loo_recall(scores_xgboost, X, ids, seeds_premium_ids, K_eval)

    print(f"  Baseline manual: Recall@{K_eval} = {r_baseline:.3f}  (posiciones: {pos_b})")
    print(f"  Coseno + emb:    Recall@{K_eval} = {r_coseno:.3f}  (posiciones: {pos_c})")
    print(f"  XGBoost PU:      Recall@{K_eval} = {r_xgb:.3f}  (posiciones: {pos_x})")
    print()

## 8 · Resumen comparativo

In [ ]:
r_baseline_20, _ = loo_recall(scores_baseline, X, ids, seeds_premium_ids, 20, es_baseline=True)
r_coseno_20, _ = loo_recall(scores_coseno, X, ids, seeds_premium_ids, 20)
r_xgb_20, _ = loo_recall(scores_xgboost, X, ids, seeds_premium_ids, 20)
r_baseline_100, _ = loo_recall(scores_baseline, X, ids, seeds_premium_ids, 100, es_baseline=True)
r_coseno_100, _ = loo_recall(scores_coseno, X, ids, seeds_premium_ids, 100)
r_xgb_100, _ = loo_recall(scores_xgboost, X, ids, seeds_premium_ids, 100)

N = len(gold)
resumen = pd.DataFrame([
    {"Modelo": "Baseline manual (filtro)", "Recall@20": r_baseline_20,
     "Recall@100": r_baseline_100, "Lift@100": r_baseline_100 / (100/N) if r_baseline_100 > 0 else 0},
    {"Modelo": "Coseno + embedding (551 feat)", "Recall@20": r_coseno_20,
     "Recall@100": r_coseno_100, "Lift@100": r_coseno_100 / (100/N) if r_coseno_100 > 0 else 0},
    {"Modelo": "XGBoost PU (551 feat)", "Recall@20": r_xgb_20,
     "Recall@100": r_xgb_100, "Lift@100": r_xgb_100 / (100/N) if r_xgb_100 > 0 else 0},
])
print(resumen.to_string(index=False))
resumen

## 9 · Visualización — posición media del seed apartado

In [ ]:
_, pos_baseline = loo_recall(scores_baseline, X, ids, seeds_premium_ids, 100, es_baseline=True)
_, pos_coseno = loo_recall(scores_coseno, X, ids, seeds_premium_ids, 100)
_, pos_xgb = loo_recall(scores_xgboost, X, ids, seeds_premium_ids, 100)

fig, ax = plt.subplots(figsize=(10, 5))
modelos = ["Baseline\nmanual", "Coseno\n+ embedding", "XGBoost\n(PU)"]
posiciones = [pos_baseline, pos_coseno, pos_xgb]
colores = ["#94a3b8", "#3b82f6", "#f97316"]

for i, (m, pos, col) in enumerate(zip(modelos, posiciones, colores)):
    ax.scatter([i]*len(pos), pos, color=col, s=100, alpha=0.7, edgecolors="white", linewidth=1.5)

ax.set_xticks(range(len(modelos)))
ax.set_xticklabels(modelos)
ax.set_ylabel("Posición del seed apartado (log)")
ax.set_title("Leave-One-Out: donde queda cada seed cuando la apartas\n(mas bajo = mejor)")
ax.axhline(y=20, color="green", linestyle=":", alpha=0.5, label="Top 20")
ax.axhline(y=100, color="orange", linestyle=":", alpha=0.5, label="Top 100")
ax.legend(loc="upper right")
ax.set_yscale("log")
ax.grid(alpha=0.3)
plt.tight_layout()

figpath = ROOT / "docs" / "figures"
figpath.mkdir(parents=True, exist_ok=True)
plt.savefig(figpath / "04_comparativa_modelos.png", dpi=150, bbox_inches="tight")
plt.show()

## 10 · Conclusiones honestas

### Lectura de los resultados

**Métricas LOO con 5 seeds premium:**

| Modelo | Recall@20 | Recall@100 |
|---|---|---|
| Baseline manual (filtro binario) | 0.600 | 1.000 |
| Coseno + embedding | 0.000 | 0.600 |
| XGBoost PU | 0.200 | 0.400 |

### Por qué el baseline "gana" pero es un espejismo

El baseline manual devuelve un **conjunto fijo de 43 restaurantes** que cumplen las 3 reglas (premium/alta_gama + web + digital activa). Los 5 seeds cumplen esas reglas por diseño, así que están dentro del filtro. Los "puestos" [6, 7, 19, 22, 25] son artificiales (orden por índice dentro de los 43 empatados a score=1).

**Recall@100 = 1.0 en baseline no significa que rankee bien**, significa que "los 5 seeds están dentro de mis 43 restaurantes filtrados". No aporta ninguna capacidad de descubrimiento: cualquier restaurante fuera del filtro (con web pero posicionamiento `medio`, o con presencia digital `básica`) queda invisible aunque semánticamente encaje con el ICP.

### Por qué el coseno gana en utilidad práctica

**Coseno + embedding: Recall@100 = 0.60** — rankea los 1683 restaurantes del universo entero. Descubre leads que:
- Sandó en posición 36
- Dray Martina en 39  
- AskuaBarra en 85

Estas 3 seeds caen dentro del **top 6% del universo** cuando se dejan fuera del centroide, lo que confirma que el modelo captura la firma vectorial del ICP más allá de las 3 reglas duras.

### XGBoost como benchmark

**XGBoost PU: Recall@20 = 0.20 con posición 1 para una seed.** El modelo mete una seed en el TOP 1 absoluto (el más parecido de los 1683), lo cual es notable. Sin embargo, en Recall@100 (0.40) queda por debajo del coseno. XGBoost es más "confiado" con los positivos claros pero menos exhaustivo con la exploración semántica.

### Sobre el embedding de resumen_llm

Comparado con el mismo modelo sin embedding (167 features estructuradas), Recall@100 apenas cambia (~0.60). El embedding **no aporta ganancia cuantitativa medible con 5 seeds premium tan homogéneas estructuralmente**, pero queda integrado en el sistema para:
- Escalar a 20-50 seeds futuras (donde el ruido estructural aumentará y el embedding aportará)  
- Habilitar explicabilidad semántica en la app (descomposición del score por bloques)

### Decisión: coseno con embedding como modelo en producción

**Ventajas frente al baseline manual:**
- Rankea el universo entero (1683), no solo un subconjunto filtrado (43)
- Score continuo interpretable ("este lead se parece al X% a tu ICP")
- Descubre leads fuera de las reglas duras que semánticamente encajan
- Escalable sin reentrenar (basta con actualizar el centroide al añadir seeds)

**Ventajas frente a XGBoost:**
- No requiere muestreo de negativos sintéticos (sesgo eliminado)
- Interpretable con SHAP-like decomposition por bloques
- Recall@100 superior (0.60 vs 0.40)

**XGBoost queda como benchmark comparativo, no como modelo principal.** El baseline manual queda como referencia de mínimos (lo que cualquier comercial haría sin herramienta).
